In [1]:
# ============================================================
# FUNCTION 6 - WEEK 13 FINAL-WEEK BAYESIAN OPTIMISATION
# Run from inside week13/
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)


# ------------------------------------------------------------
# 1. Load Week 13 cumulative data
# ------------------------------------------------------------

X = np.load(
    "function6/initial_inputs.npy"
)

Y = np.load(
    "function6/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

tree = cKDTree(X)

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Recent local evidence
# ------------------------------------------------------------

week11_x = np.array([
    0.444632,
    0.410696,
    0.586583,
    0.711960,
    0.118032
])

week11_y = (
    -0.20392262491163674
)

week12_x = np.array([
    0.444319,
    0.410445,
    0.594906,
    0.717081,
    0.120120
])

week12_y = (
    -0.260502789774765
)

print("\n================================")
print("RECENT LOCAL EVIDENCE")
print("================================")

print("\nIncumbent:")
print(
    best_x,
    "->",
    best_y
)

print("\nWeek 11:")
print(
    week11_x,
    "->",
    week11_y
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        week11_x - best_x
    )
)

print("\nWeek 12:")
print(
    week12_x,
    "->",
    week12_y
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        week12_x - best_x
    )
)

print(
    "\nWeek 12 change from Week 11:"
)

print(
    week12_x - week11_x
)

print(
    "\nWeek 12 objective change from Week 11:"
)

print(
    week12_y - week11_y
)


# ------------------------------------------------------------
# 3. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(5) * 0.2,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=15,
    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0 / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)

print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)

print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Final week: keep search tightly local
empirical_cap = min(
    1.25 * nearest_distance,
    0.035
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 6. ARD-informed trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.15 * lengthscales,
    0.010,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 13 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)

print(
    trust_half_width
)

print(
    "\nLower:"
)

print(
    lower
)

print(
    "\nUpper:"
)

print(
    upper
)


# ------------------------------------------------------------
# 7. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        300000,
        5
    )
)

# Portal precision BEFORE duplicate filtering
candidates = np.round(
    candidates,
    6
)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 8. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 9. EI - diagnostic only
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI - DIAGNOSTIC ONLY")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 10. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"\nbeta={beta}"
    )

    print(
        "candidate =",
        candidates[idx]
    )

    print(
        "mean =",
        mu[idx]
    )

    print(
        "std =",
        sigma[idx]
    )

    print(
        "distance =",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 12. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 13. Axis-profile diagnostics
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTICS")
print("================================")

for j in range(5):

    values = np.linspace(
        lower[j],
        upper[j],
        40000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile = np.round(
        profile,
        6
    )

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    profile_valid = profile[
        profile_distance > 0.01
    ]

    if len(profile_valid) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        profile_valid,
        return_std=True
    )

    idx = np.argmax(
        profile_mu
    )

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        profile_valid[idx]
    )

    print(
        "mean =",
        profile_mu[idx]
    )

    print(
        "std =",
        profile_sigma[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            profile_valid[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 14. Week 12 mirror diagnostic
# ------------------------------------------------------------
#
# Week 12 sits on the side of the incumbent that performed
# poorly. Mirror the Week 12 displacement across the incumbent.
# ------------------------------------------------------------

week12_step = (
    week12_x
    - best_x
)

mirror_candidate = (
    best_x
    - week12_step
)

mirror_candidate = np.clip(
    mirror_candidate,
    0.0,
    1.0
)

mirror_candidate = np.round(
    mirror_candidate,
    6
)

mirror_dist, mirror_idx = tree.query(
    mirror_candidate.reshape(1, -1),
    k=1
)

print("\n================================")
print("WEEK 12 MIRROR DIAGNOSTIC")
print("================================")

print(
    "Mirror candidate =",
    mirror_candidate
)

print(
    "Nearest observed distance =",
    mirror_dist[0]
)

print(
    "Nearest observed point =",
    X[mirror_idx[0]]
)

if mirror_dist[0] > 0.01:

    mirror_mu, mirror_sigma = gp.predict(
        mirror_candidate.reshape(1, -1),
        return_std=True
    )

    print(
        "mean =",
        mirror_mu[0]
    )

    print(
        "std =",
        mirror_sigma[0]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            mirror_candidate
            - best_x
        )
    )

else:

    print(
        "Mirror candidate is too close to an existing observation."
    )


# ------------------------------------------------------------
# 15. Direction AWAY from failed Week 12 point
# ------------------------------------------------------------
#
# Construct the empirical direction from Week 12 back toward
# the incumbent, then continue beyond the incumbent.
#
# This is a systematic final-week test of the opposite side of
# the local basin rather than continuing through the region
# where Week 12 performed poorly.
# ------------------------------------------------------------

away_direction = (
    best_x
    - week12_x
)

away_norm = np.linalg.norm(
    away_direction
)

if away_norm > 0:

    away_direction = (
        away_direction
        / away_norm
    )

    positive_limits = []

    for j in range(5):

        if away_direction[j] > 0:

            positive_limits.append(
                (
                    upper[j]
                    - best_x[j]
                )
                / away_direction[j]
            )

        elif away_direction[j] < 0:

            positive_limits.append(
                (
                    lower[j]
                    - best_x[j]
                )
                / away_direction[j]
            )

    t_max = min(
        positive_limits
    )

    t_values = np.linspace(
        0.0,
        t_max,
        60000
    )

    away_candidates = (
        best_x
        + t_values[:, None]
        * away_direction
    )

    away_candidates = np.round(
        away_candidates,
        6
    )

    away_distance, _ = tree.query(
        away_candidates,
        k=1
    )

    away_valid = away_candidates[
        away_distance > 0.01
    ]

    print("\n================================")
    print("AWAY-FROM-WEEK-12 PROFILE")
    print("================================")

    if len(away_valid) > 0:

        away_mu, away_sigma = gp.predict(
            away_valid,
            return_std=True
        )

        away_idx = np.argmax(
            away_mu
        )

        print(
            "Best candidate =",
            away_valid[away_idx]
        )

        print(
            "mean =",
            away_mu[away_idx]
        )

        print(
            "std =",
            away_sigma[away_idx]
        )

        print(
            "distance from incumbent =",
            np.linalg.norm(
                away_valid[away_idx]
                - best_x
            )
        )

    else:

        print(
            "No admissible candidates."
        )


# ------------------------------------------------------------
# 16. Smallest admissible point on away-from-Week12 line
# ------------------------------------------------------------
#
# Useful final-week comparison:
# if GP improvement is weak, we may prefer the closest legal
# point on the empirically safer side of the incumbent.
# ------------------------------------------------------------

if away_norm > 0 and len(away_valid) > 0:

    incumbent_distances = np.linalg.norm(
        away_valid
        - best_x,
        axis=1
    )

    closest_idx = np.argmin(
        incumbent_distances
    )

    closest_candidate = (
        away_valid[closest_idx]
    )

    closest_mu, closest_sigma = gp.predict(
        closest_candidate.reshape(1, -1),
        return_std=True
    )

    print("\n================================")
    print("CLOSEST ADMISSIBLE AWAY-SIDE POINT")
    print("================================")

    print(
        "candidate =",
        closest_candidate
    )

    print(
        "mean =",
        closest_mu[0]
    )

    print(
        "std =",
        closest_sigma[0]
    )

    print(
        "distance from incumbent =",
        incumbent_distances[closest_idx]
    )

DATA
X shape: (32, 5)
Y shape: (32,)

Current best:
[0.442929 0.409333 0.63183  0.7398   0.129381] -> -0.19517557780237

Y range:
min = -2.5711696316081234
max = -0.19517557780237
std = 0.6738463599163148

RECENT LOCAL EVIDENCE

Incumbent:
[0.442929 0.409333 0.63183  0.7398   0.129381] -> -0.19517557780237

Week 11:
[0.444632 0.410696 0.586583 0.71196  0.118032] -> -0.20392262491163674
distance from incumbent = 0.05436832154849

Week 12:
[0.444319 0.410445 0.594906 0.717081 0.12012 ] -> -0.260502789774765
distance from incumbent = 0.04436743740627803

Week 12 change from Week 11:
[-0.000313 -0.000251  0.008323  0.005121  0.002088]

Week 12 objective change from Week 11:
-0.05658016486312825

GP FIT

Fitted kernel:
1.42**2 * Matern(length_scale=[0.911, 1.08, 1.5, 1.04, 1.16], nu=2.5) + WhiteKernel(noise_level=0.00252)

ARD lengthscales:
[0.91095004 1.07590796 1.49946354 1.04420071 1.16298482]

Normalised inverse-lengthscale sensitivity:
[0.24331648 0.20601127 0.14781897 0.21226682 0.190

In [2]:
# ============================================================
# FINAL FUNCTION 6 - WEEK 13 SELECTION
# ============================================================
#
# FINAL CAPSTONE QUERY
#
# Evidence:
# - The incumbent remains the best observed point.
# - Week 11 was close to the incumbent, but Week 12 showed
#   substantial deterioration within the lower-x3/lower-x4
#   region.
#
# - The unrestricted posterior-mean and UCB candidates return
#   toward that same region, so they are too aggressive for
#   the final query.
#
# - The x3-only and x4-only profile winners also require much
#   larger displacements from the incumbent:
#
#       x3 move: ~0.035
#       x4 move: ~0.031
#
# - Among the smallest admissible one-coordinate changes,
#   x1 gives the strongest posterior mean:
#
#       [0.452930, 0.409333, 0.631830, 0.739800, 0.129381]
#
# - This keeps x2, x3, x4 and x5 exactly at their proven
#   incumbent values and changes only x1 by 0.010001.
#
# Final-week strategy:
# conservative exploitation around the observed incumbent.
# ============================================================

week13_candidate = np.array([
    0.452930,
    0.409333,
    0.631830,
    0.739800,
    0.129381
])

print("================================")
print("FUNCTION 6 - WEEK 13 FINAL")
print("================================")

print("\nCandidate:")
print(week13_candidate)


# ------------------------------------------------------------
# GP prediction
# ------------------------------------------------------------

final_mu, final_sigma = gp.predict(
    week13_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])


# ------------------------------------------------------------
# Distance from incumbent
# ------------------------------------------------------------

distance_from_best = np.linalg.norm(
    week13_candidate - best_x
)

print("\nDistance from current best:")
print(distance_from_best)


# ------------------------------------------------------------
# Duplicate-rule verification
# ------------------------------------------------------------

nearest_dist, nearest_idx = tree.query(
    week13_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

print("\nDuplicate rule satisfied:")
print(
    nearest_dist[0] > 0.01
)


# ------------------------------------------------------------
# Portal format
# ------------------------------------------------------------

portal = "-".join(
    f"{x:.6f}"
    for x in week13_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 6 - WEEK 13 FINAL

Candidate:
[0.45293  0.409333 0.63183  0.7398   0.129381]

Predicted mean:
-0.23898610274215715

Predicted std:
0.03761241781296123

Distance from current best:
0.010000999999999982

Nearest observed-point distance:
0.010000999999999982

Nearest observed point:
[0.442929 0.409333 0.63183  0.7398   0.129381]

Duplicate rule satisfied:
True

Portal format:
0.452930-0.409333-0.631830-0.739800-0.129381
